In [1]:
from collections import defaultdict, Counter
import itertools
import math


In [2]:
training_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
    [("Dogs", "NOUN"), ("bark", "VERB")],
    [("Cats", "NOUN"), ("meow", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
]


In [3]:
def train_hmm(training_data, smoothing=1.0):
    tags = set()
    words = set()
    tag_counts = Counter()

    tag_starts = Counter()
    tag_transitions = defaultdict(Counter)
    tag_emissions = defaultdict(Counter)

    for sentence in training_data:
        first_word, first_tag = sentence[0]
        tag_starts[first_tag] += 1
        tag_counts[first_tag] += 1
        tag_emissions[first_tag][first_word] += 1
        tags.add(first_tag)
        words.add(first_word)

        for i in range(1, len(sentence)):
            word, tag = sentence[i]
            prev_tag = sentence[i-1][1]
            tag_transitions[prev_tag][tag] += 1
            tag_emissions[tag][word] += 1
            tag_counts[tag] += 1
            tags.add(tag)
            words.add(word)

    num_tags = len(tags)
    num_words = len(words)
    total_sentences = len(training_data)

    initial_probs = {
        t: (tag_starts[t] + smoothing) / (total_sentences + smoothing * num_tags)
        for t in tags
    }

    transition_probs = {
        pt: {
            ct: (tag_transitions[pt][ct] + smoothing) /
                (tag_counts[pt] + smoothing * num_tags)
            for ct in tags
        }
        for pt in tags
    }

    emission_probs = {
        t: {
            w: (tag_emissions[t][w] + smoothing) /
               (tag_counts[t] + smoothing * (num_words + 1))
            for w in words
        }
        for t in tags
    }

    return tags, words, tag_counts, initial_probs, transition_probs, emission_probs


tags, words, tag_counts, initial_probs, transition_probs, emission_probs = train_hmm(training_data)
print("Training complete. Tags:", tags)


Training complete. Tags: {'DET', 'NOUN', 'ADJ', 'VERB', 'ADP'}


In [4]:
def brute_force_decode(sentence):
    best_sequence = None
    best_log_prob = -float("inf")

    unk_emission = {
        t: 1.0 / (tag_counts[t] + (len(words) + 1))
        for t in tags
    }

    for seq in itertools.product(tags, repeat=len(sentence)):
        log_prob = 0.0

        for i, word in enumerate(sentence):
            tag = seq[i]
            emission = emission_probs[tag].get(word, unk_emission[tag])

            if i == 0:
                log_prob += math.log(initial_probs[tag])
            else:
                log_prob += math.log(transition_probs[seq[i-1]][tag])

            log_prob += math.log(emission)

        if log_prob > best_log_prob:
            best_log_prob = log_prob
            best_sequence = seq

    return list(best_sequence)


In [5]:
test_sentences = [
    ["The", "dog", "barks"],
    ["A", "cat", "sleeps"],
    ["The", "big", "dog", "runs"],
    ["Dogs", "bark"],
]

for s in test_sentences:
    print(s, "->", brute_force_decode(s))


['The', 'dog', 'barks'] -> ['DET', 'NOUN', 'VERB']
['A', 'cat', 'sleeps'] -> ['DET', 'NOUN', 'VERB']
['The', 'big', 'dog', 'runs'] -> ['DET', 'ADJ', 'NOUN', 'VERB']
['Dogs', 'bark'] -> ['NOUN', 'VERB']


In [6]:
def viterbi_decode(sentence):
    V = []          # dynamic programming table
    backpointer = []

    # Initialization
    V0 = {}
    bp0 = {}
    for tag in tags:
        emission = emission_probs[tag].get(
            sentence[0],
            1.0 / (tag_counts[tag] + (len(words) + 1))
        )
        V0[tag] = math.log(initial_probs[tag]) + math.log(emission)
        bp0[tag] = None

    V.append(V0)
    backpointer.append(bp0)

    # Recursion
    for t in range(1, len(sentence)):
        Vt = {}
        bpt = {}
        for curr_tag in tags:
            best_prev_tag = None
            best_score = -float("inf")

            emission = emission_probs[curr_tag].get(
                sentence[t],
                1.0 / (tag_counts[curr_tag] + (len(words) + 1))
            )

            for prev_tag in tags:
                score = (
                    V[t-1][prev_tag]
                    + math.log(transition_probs[prev_tag][curr_tag])
                    + math.log(emission)
                )

                if score > best_score:
                    best_score = score
                    best_prev_tag = prev_tag

            Vt[curr_tag] = best_score
            bpt[curr_tag] = best_prev_tag

        V.append(Vt)
        backpointer.append(bpt)

    # Termination
    last_tag = max(V[-1], key=V[-1].get)
    best_path = [last_tag]

    # Backtracking
    for t in range(len(sentence)-1, 0, -1):
        last_tag = backpointer[t][last_tag]
        best_path.insert(0, last_tag)

    return best_path


In [7]:
for s in test_sentences:
    print("Sentence:", s)
    print("Brute force:", brute_force_decode(s))
    print("Viterbi    :", viterbi_decode(s))
    print()


Sentence: ['The', 'dog', 'barks']
Brute force: ['DET', 'NOUN', 'VERB']
Viterbi    : ['DET', 'NOUN', 'VERB']

Sentence: ['A', 'cat', 'sleeps']
Brute force: ['DET', 'NOUN', 'VERB']
Viterbi    : ['DET', 'NOUN', 'VERB']

Sentence: ['The', 'big', 'dog', 'runs']
Brute force: ['DET', 'ADJ', 'NOUN', 'VERB']
Viterbi    : ['DET', 'ADJ', 'NOUN', 'VERB']

Sentence: ['Dogs', 'bark']
Brute force: ['NOUN', 'VERB']
Viterbi    : ['NOUN', 'VERB']

